# GARCH(1,1) by Maximum Likelihood

EWMA (09/02) tracks volatility well but has two weaknesses: its decay factor is fixed by convention, and its forecasts never revert to a
long-run level. GARCH(1,1) (Bollerslev 1986) adds a constant to the EWMA recursion, which gives volatility a long-run mean to which
forecasts converge, and estimates all parameters from the data by maximum likelihood. This notebook derives the model's properties,
writes its likelihood, estimates it with Gaussian and Student-$t$ innovations, computes standard errors, and uses the fitted model
for volatility forecasts at different horizons and for value at risk.

## 0. Math link

- [Introduction to maximum likelihood](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/05_point_estimation/01_intro_to_maximum_likelihood.ipynb)
  and [MLE for the Gaussian](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/05_point_estimation/03_mle_gaussian.ipynb):
  the GARCH likelihood is a product of normal densities whose variance changes every day.
- [Geometric sequences and limits](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/01_limits/00_limits_of_sequences.ipynb):
  forecasts converge to the long-run variance at the rate $(\alpha+\beta)^h$.
- EWMA, QLIKE and volatility clustering from [09/02](./02_volatility_clustering_ewma_volatility.ipynb); Ljung–Box from
  [09/01](./01_random_walk_and_autocorrelation.ipynb).

## 1. Introduction

After a market crash, implied volatility for the next month is far above its long-run level, but implied volatility for the next
two years is only slightly above it: markets expect turbulence to fade. EWMA cannot produce such a term structure, GARCH can.

After this notebook you can:

- write the GARCH(1,1) recursion and derive its long-run variance, persistence and half-life;
- forecast volatility at any horizon and explain the mean reversion;
- write the Gaussian and Student-$t$ log-likelihood and maximize it numerically;
- compute standard errors from the Hessian and check them by simulation;
- diagnose a fitted model with standardized residuals;
- compare GARCH with EWMA by likelihood and by forecasts.

## 2. Theory

### The model

$$
r_t = \sigma_t z_t, \qquad \sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta\sigma_{t-1}^2, \qquad z_t \text{ i.i.d. with mean 0, variance 1}
$$

with $\omega > 0$, $\alpha \ge 0$, $\beta \ge 0$ and $\alpha + \beta < 1$. $\alpha$ is the reaction to yesterday's shock, $\beta$ the memory of yesterday's variance.

### Long-run variance and persistence

Taking expectations of the recursion in the stationary state ($E[r_{t-1}^2] = E[\sigma_{t-1}^2] = \bar\sigma^2$):

$$
\bar\sigma^2 = \frac{\omega}{1 - \alpha - \beta}
$$

The recursion can be rewritten as $\sigma_t^2 - \bar\sigma^2 = \alpha(r_{t-1}^2 - \bar\sigma^2) + \beta(\sigma_{t-1}^2 - \bar\sigma^2)$: deviations from the long-run level decay with factor
$\alpha + \beta$, the **persistence**, with half-life $\ln 0.5/\ln(\alpha+\beta)$.

### Forecasts

Since $E_t[r_{t+j}^2] = E_t[\sigma_{t+j}^2]$,

$$
E_t[\sigma_{t+h}^2] = \bar\sigma^2 + (\alpha+\beta)^{h-1}\big(\sigma_{t+1}^2 - \bar\sigma^2\big)
$$

and the variance of the $H$-day return is the sum of the daily forecasts, $\sum_{h=1}^H E_t[\sigma_{t+h}^2]$. From a high-volatility state the average volatility over
longer horizons falls toward $\bar\sigma$; from a calm state it rises. This is the **term structure of volatility**.

EWMA is the special case $\omega = 0$, $\alpha + \beta = 1$ (IGARCH): no long-run level, flat forecasts.

### Fat tails

Even with normal $z_t$, GARCH returns have kurtosis

$$
\kappa = 3\,\frac{1 - (\alpha+\beta)^2}{1 - (\alpha+\beta)^2 - 2\alpha^2} > 3
$$

(finite if $(\alpha+\beta)^2 + 2\alpha^2 < 1$). Real data usually need fat-tailed $z_t$ as well, e.g. Student-$t$.

### The likelihood

Given parameters $\theta = (\omega, \alpha, \beta)$ and a starting value $\sigma_1^2$ (e.g. the sample variance), the recursion gives $\sigma_t^2(\theta)$ for every day. Conditionally on the
past, $r_t \sim N(0, \sigma_t^2)$, so the log-likelihood is a sum of normal log-densities:

$$
\ell(\theta) = -\frac12\sum_{t=1}^n\Big[\ln(2\pi) + \ln\sigma_t^2(\theta) + \frac{r_t^2}{\sigma_t^2(\theta)}\Big]
$$

(the QLIKE loss of 09/02, up to constants). With standardized Student-$t$ innovations with $\nu$ degrees of freedom:

$$
\ell(\theta, \nu) = \sum_t\Big[\ln\Gamma\big(\tfrac{\nu+1}{2}\big) - \ln\Gamma\big(\tfrac\nu2\big) - \tfrac12\ln\big(\pi(\nu-2)\sigma_t^2\big) - \tfrac{\nu+1}{2}\ln\Big(1 + \frac{r_t^2}{(\nu-2)\sigma_t^2}\Big)\Big]
$$

There is no closed form for the maximum; it is found numerically. Maximizing the Gaussian likelihood when $z_t$ is not normal is
**quasi-maximum likelihood** (QMLE): the estimates are still consistent, but the standard errors need a robust ("sandwich") correction.

### Standard errors

Near the maximum, $\ell$ is approximately quadratic. The covariance matrix of $\hat\theta$ is approximately the inverse of the negative Hessian
$-\nabla^2\ell(\hat\theta)$ (the observed Fisher information), computed numerically by finite differences.

### Variance targeting

Fixing $\omega = \hat\sigma^2_{\text{sample}}(1 - \alpha - \beta)$ leaves two parameters to estimate and guarantees that the model's long-run volatility equals the sample's.

## 3. Worked Examples

Parameters $\omega = 2\cdot 10^{-6}$, $\alpha = 0.08$, $\beta = 0.90$ (daily).

### Example 1: long-run level and half-life

$\bar\sigma^2 = 2\cdot 10^{-6}/0.02 = 10^{-4}$, so $\bar\sigma = 1\%$ daily, $15.9\%$ annual. Persistence 0.98, half-life $\ln 0.5/\ln 0.98 = 34.3$ days.

### Example 2: one update

Today's variance $1.5\cdot 10^{-4}$ (1.22% daily) and today's return −2.5%:

$$
\sigma_{t+1}^2 = 2\cdot 10^{-6} + 0.08\cdot 6.25\cdot 10^{-4} + 0.90\cdot 1.5\cdot 10^{-4} = 1.87\cdot 10^{-4}, \qquad \sigma_{t+1} = 1.37\%
$$

### Example 3: forecasts

In 20 days: $10^{-4} + 0.98^{19}(1.87 - 1)\cdot 10^{-4} = 1.59\cdot 10^{-4}$, volatility 1.26%. Variance of the 20-day return:

$$
\sum_{h=1}^{20}\big[10^{-4} + 0.98^{h-1}\cdot 0.87\cdot 10^{-4}\big] = 20\cdot 10^{-4} + 0.87\cdot 10^{-4}\cdot\frac{1 - 0.98^{20}}{0.02} = 0.00345
$$

20-day volatility 5.87%, against $\sqrt{20}\cdot 1.37\% = 6.12\%$ if the current level persisted (EWMA).

### Example 4: kurtosis

$(\alpha+\beta)^2 = 0.9604$, $2\alpha^2 = 0.0128$: $\kappa = 3\cdot 0.0396/(0.0396 - 0.0128) = 4.43$ with normal innovations.

## 4. Finance Application

### Where it matters

- **Risk management.** GARCH-based VaR and expected shortfall react to conditions like EWMA, and multi-day forecasts revert toward the
  long-run level instead of extrapolating today's volatility.
- **Option pricing and volatility trading.** GARCH term structures are compared with implied volatility curves; a gap is a view on
  future volatility.
- **Volatility targeting over longer horizons.** Funds rebalancing monthly can use the 21-day GARCH forecast rather than today's EWMA.
- **Stress testing.** Simulating from a fitted GARCH reproduces clustering and fat tails, unlike i.i.d. simulation.

### The usual misreadings

- "$\alpha + \beta = 0.99$, so volatility is almost a random walk." Shocks are persistent (half-life 69 days), but the model still has a long-run level;
  EWMA does not.
- "The model fits, so the tails are normal." Standardized residuals of equity GARCH models typically still have fat tails; check them.
- "Parameters are precise because $n$ is large." $\alpha$ and $\beta$ are strongly negatively correlated in estimation; their sum is precise, the split less so.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from scipy.signal import lfilter
from scipy.special import gammaln

rng = np.random.default_rng(42)
trading_days = 252

### Data

Twenty years of daily returns simulated from GARCH(1,1) with the parameters of the worked examples and Student-$t$ innovations with 6
degrees of freedom.

In [ ]:
omega0, alpha0, beta0, nu0 = 2e-6, 0.08, 0.90, 6
n = 20 * trading_days

def simulate_garch(omega, alpha, beta, n_, rng_, nu=None):
    z_ = rng_.standard_normal(n_) if nu is None else rng_.standard_t(nu, n_) / np.sqrt(nu / (nu - 2))
    s2 = np.empty(n_)
    r_ = np.empty(n_)
    s2[0] = omega / (1 - alpha - beta)
    for t in range(n_):
        if t > 0:
            s2[t] = omega + alpha * r_[t - 1] ** 2 + beta * s2[t - 1]
        r_[t] = np.sqrt(s2[t]) * z_[t]
    return r_, s2

r, s2_true = simulate_garch(omega0, alpha0, beta0, n, rng, nu0)
print(f"annualized vol {r.std() * np.sqrt(252):.2%}, kurtosis {stats.kurtosis(r, fisher=False):.2f}")

### Likelihood and estimation

The variance recursion is a linear filter, so `scipy.signal.lfilter` computes it without a Python loop. Parameters are optimized on
transformed scales so that the constraints hold automatically: $\omega = e^{a}$, $(\alpha, \beta)$ from a softmax that keeps $\alpha + \beta < 1$, and $\nu = 2 + e^{c}$.
The quasi-Newton method BFGS does the maximization; a derivative-free simplex search (Nelder–Mead) tends to stop early on this flat, curved
likelihood surface.

In [ ]:
def garch_variance(params, r_):
    omega, alpha, beta = params
    x = np.empty_like(r_)
    x[0] = r_.var()                                   # sigma_1^2 = sample variance
    x[1:] = omega + alpha * r_[:-1] ** 2
    return lfilter([1.0], [1.0, -beta], x)            # s2[t] = x[t] + beta * s2[t-1]


def loglik_normal(params, r_):
    s2 = garch_variance(params, r_)
    return -0.5 * np.sum(np.log(2 * np.pi) + np.log(s2) + r_**2 / s2)


def loglik_t(params, nu, r_):
    s2 = garch_variance(params, r_)
    return np.sum(gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2) * s2)
                  - (nu + 1) / 2 * np.log1p(r_**2 / ((nu - 2) * s2)))


def to_params(u):
    # u = (log omega, u_alpha, u_beta) -> alpha, beta > 0 with alpha + beta < 1
    e = np.exp([u[1], u[2], 0.0])
    return np.array([np.exp(u[0]), e[0] / e.sum(), e[1] / e.sum()])


def from_params(p):
    omega, alpha, beta = p
    rest = 1 - alpha - beta
    return np.array([np.log(omega), np.log(alpha / rest), np.log(beta / rest)])


def fit_garch(r_, dist="normal"):
    start = from_params([r_.var() * 0.05, 0.05, 0.90])
    if dist == "normal":
        obj = lambda u: -loglik_normal(to_params(u), r_)
        res = minimize(obj, start, method="BFGS")
        return to_params(res.x), -res.fun
    obj = lambda u: -loglik_t(to_params(u[:3]), 2 + np.exp(u[3]), r_)
    res = minimize(obj, np.r_[start, np.log(4.0)], method="BFGS")
    return np.r_[to_params(res.x[:3]), 2 + np.exp(res.x[3])], -res.fun


p_norm, ll_norm = fit_garch(r, "normal")
p_t, ll_t = fit_garch(r, "t")
fit_table = pd.DataFrame({"true": [omega0, alpha0, beta0, nu0], "Gaussian (QMLE)": np.r_[p_norm, np.nan], "Student-t": p_t},
                         index=["omega", "alpha", "beta", "nu"])
fit_table.loc["alpha + beta"] = [alpha0 + beta0, p_norm[1] + p_norm[2], p_t[1] + p_t[2]]
fit_table.loc["long-run vol (annual)"] = [np.sqrt(omega0 / (1 - alpha0 - beta0) * 252),
                                          np.sqrt(p_norm[0] / (1 - p_norm[1] - p_norm[2]) * 252),
                                          np.sqrt(p_t[0] / (1 - p_t[1] - p_t[2]) * 252)]
fit_table.loc["log-likelihood"] = [np.nan, ll_norm, ll_t]
fit_table

Both fits recover the parameters: persistence 0.981 against a true 0.980, long-run volatility 15.8–15.9% against 15.9%. The Student-$t$ model
estimates $\nu = 5.4$ (true 6) and raises the log-likelihood by 167 points over the Gaussian one, a large improvement for one extra parameter.

### Standard errors from the Hessian

In [ ]:
def numerical_hessian(f, x, eps=None):
    x = np.asarray(x, dtype=float)
    k = x.size
    eps = 1e-4 * np.maximum(np.abs(x), 1e-6) if eps is None else eps
    H = np.empty((k, k))
    for i in range(k):
        for j in range(i, k):
            ei, ej = np.zeros(k), np.zeros(k)
            ei[i], ej[j] = eps[i], eps[j]
            H[i, j] = H[j, i] = (f(x + ei + ej) - f(x + ei - ej) - f(x - ei + ej) + f(x - ei - ej)) / (4 * eps[i] * eps[j])
    return H

H = numerical_hessian(lambda p: loglik_t(p[:3], p[3], r), p_t)
cov_t = np.linalg.inv(-H)
se_t = np.sqrt(np.diag(cov_t))
corr_ab = cov_t[1, 2] / (se_t[1] * se_t[2])
se_sum = np.sqrt(cov_t[1, 1] + cov_t[2, 2] + 2 * cov_t[1, 2])
pd.DataFrame({"estimate": p_t, "SE": se_t, "t-stat": p_t / se_t}, index=["omega", "alpha", "beta", "nu"]).round(6), \
    f"corr(alpha_hat, beta_hat) = {corr_ab:.2f}; SE(alpha + beta) = {se_sum:.4f}"

All parameters are many standard errors from zero. $\hat\alpha$ and $\hat\beta$ have correlation −0.86: their sum (standard error 0.0056) is much better
determined than either one.

### Do the standard errors match the sampling variation?

Fit the Student-$t$ GARCH to 60 independently simulated 20-year histories.

In [ ]:
rng_mc = np.random.default_rng(7)
mc = np.array([fit_garch(simulate_garch(omega0, alpha0, beta0, n, rng_mc, nu0)[0], "t")[0] for _ in range(60)])
pd.DataFrame({"true": [omega0, alpha0, beta0, nu0], "mean estimate": mc.mean(axis=0), "SD across histories": mc.std(axis=0),
              "Hessian SE (one history)": se_t}, index=["omega", "alpha", "beta", "nu"]).round(6)

The Hessian standard errors from one history (0.009 for $\alpha$, 0.011 for $\beta$) match the spread of estimates across 60 histories (0.010 and 0.011), and the
estimates are centred on the true values. For $\nu$ the Hessian understates the spread somewhat (0.41 against 0.56).

### Diagnostics: standardized residuals

In [ ]:
def ljung_box(x, m):
    x = x - x.mean()
    n_ = len(x)
    rho = np.array([x[k:] @ x[:-k] / (x @ x) for k in range(1, m + 1)])
    Q = n_ * (n_ + 2) * np.sum(rho**2 / (n_ - np.arange(1, m + 1)))
    return Q, stats.chi2.sf(Q, m)

s2_fit = garch_variance(p_t[:3], r)
z_hat = r / np.sqrt(s2_fit)
for name, x in [("returns", r), ("standardized residuals", z_hat)]:
    print(f"{name:23s}: kurtosis {stats.kurtosis(x, fisher=False):5.2f}, Ljung-Box(10) on squares p = {ljung_box(x**2, 10)[1]:.3f}")
print(f"theoretical kurtosis of standardized t({p_t[3]:.1f}): {3 + 6 / (p_t[3] - 4):.2f}")

The model removes the autocorrelation of squared returns (Ljung–Box p-value from 0.000 to 0.075) and lowers the kurtosis from 8.0 to 5.6. What remains
is the fat tail of the innovations; the sample kurtosis of a $t_{5.4}$ sample is itself very noisy and usually below its theoretical value of 7.2.

### GARCH vs EWMA

In [ ]:
def ewma_variance(r_, lam):
    x = np.empty_like(r_)
    x[0] = r_.var()
    x[1:] = (1 - lam) * r_[:-1] ** 2
    return lfilter([1.0], [1.0, -lam], x)

def loglik_from_s2(s2, r_, nu):
    return np.sum(gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2) * s2)
                  - (nu + 1) / 2 * np.log1p(r_**2 / ((nu - 2) * s2)))

ll_ewma = loglik_from_s2(ewma_variance(r, 0.94), r, p_t[3])
print(f"log-likelihood (t, nu = {p_t[3]:.1f}): GARCH {ll_t:.1f}, EWMA 0.94 {ll_ewma:.1f}, difference {ll_t - ll_ewma:.1f}")
rmse = lambda s2: np.sqrt(np.mean((np.sqrt(s2) - np.sqrt(s2_true)) ** 2)) * np.sqrt(252)
print(f"RMSE of conditional vol vs truth (annualized): GARCH {rmse(s2_fit):.2%}, EWMA {rmse(ewma_variance(r, 0.94)):.2%}")

GARCH beats EWMA by 39 log-likelihood points and tracks the true conditional volatility almost exactly (RMSE 0.13 points against 1.42): not
surprising, since the data come from a GARCH process, but it shows what is gained when the model fits.

### Forecasting the term structure of volatility

From the calmest and the most turbulent day in the sample, forecast the average annualized volatility over the next $H$ days.

In [ ]:
def garch_term_structure(p, s2_next, H_max):
    omega, alpha, beta = p
    v_bar = omega / (1 - alpha - beta)
    h = np.arange(1, H_max + 1)
    daily = v_bar + (alpha + beta) ** (h - 1) * (s2_next - v_bar)
    return np.sqrt(np.cumsum(daily) / h * 252)            # average annualized vol over the next H days

H_max = 252
t_hi, t_lo = int(np.argmax(s2_fit)), int(np.argmin(s2_fit[252:])) + 252
ts = {}
for lab, t0 in [("turbulent day", t_hi), ("calm day", t_lo)]:
    s2_next = p_t[0] + p_t[1] * r[t0] ** 2 + p_t[2] * s2_fit[t0]
    ts[lab] = garch_term_structure(p_t[:3], s2_next, H_max)
    print(f"{lab:14s}: next-day vol {np.sqrt(s2_next * 252):.1%}, average over 21 days {ts[lab][20]:.1%}, "
          f"over 252 days {ts[lab][-1]:.1%}  (long-run {np.sqrt(p_t[0] / (1 - p_t[1] - p_t[2]) * 252):.1%})")

From the most turbulent day, next-day volatility is 45.5% but the average over the next year only 25.0%; from the calmest day, 8.6% rises to 14.7%.
Both converge toward the long-run 15.9%. An EWMA forecast would keep 45.5% and 8.6% for the whole year.

### Value at risk with the fitted model

99% one-day VaR with the GARCH-$t$ volatility and the $t$ quantile (standardized), versus EWMA with the normal quantile.

In [ ]:
nu_hat = p_t[3]
q_t = stats.t.ppf(0.01, nu_hat) * np.sqrt((nu_hat - 2) / nu_hat)
q_n = stats.norm.ppf(0.01)
exc_garch = r[252:] < q_t * np.sqrt(s2_fit[252:])
exc_ewma = r[252:] < q_n * np.sqrt(ewma_variance(r, 0.94)[252:])
print(f"exceedance rate (expected 1%): GARCH-t {exc_garch.mean():.2%}, EWMA-normal {exc_ewma.mean():.2%}")

With the $t$ quantile and GARCH volatility, losses exceed the 99% VaR on 1.15% of days, close to the nominal 1%; the EWMA VaR with a normal quantile
is exceeded on 1.80%.

## 6. Visualizations

In [ ]:
a_grid = np.linspace(0.02, 0.16, 50)
b_grid = np.linspace(0.80, 0.97, 50)
LL = np.full((b_grid.size, a_grid.size), np.nan)
var_s = r.var()
for i, b in enumerate(b_grid):
    for j, a in enumerate(a_grid):
        if a + b < 0.999:
            LL[i, j] = loglik_normal([var_s * (1 - a - b), a, b], r)       # variance targeting
fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contour(a_grid, b_grid, LL - np.nanmax(LL), levels=[-200, -100, -50, -20, -10, -5, -2])
ax.clabel(cs, fontsize=8)
ax.plot(alpha0, beta0, "k*", markersize=12, label="true")
ax.plot(p_norm[1], p_norm[2], "ro", label="Gaussian QMLE")
ax.plot(a_grid, 1 - a_grid, color="grey", linestyle=":", label="alpha + beta = 1")
ax.set_xlim(a_grid[0], a_grid[-1])
ax.set_ylim(b_grid[0], b_grid[-1])
ax.set_title("Gaussian log-likelihood (relative to its maximum), variance targeting")
ax.set_xlabel("alpha")
ax.set_ylabel("beta")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
years = np.arange(n) / 252
ax.plot(years, np.sqrt(s2_true * 252), color="black", linewidth=1, label="true")
ax.plot(years, np.sqrt(s2_fit * 252), color="tab:red", linewidth=0.8, alpha=0.8, label="fitted GARCH-t")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Conditional volatility (annualized): true vs fitted")
ax.set_xlabel("years")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
H = np.arange(1, H_max + 1)
for lab, c in [("turbulent day", "tab:red"), ("calm day", "tab:blue")]:
    ax.plot(H, ts[lab], color=c, label=f"GARCH from the {lab}")
    ax.axhline(ts[lab][0], color=c, linestyle=":", label=f"EWMA-style flat forecast from the {lab}")
ax.axhline(np.sqrt(p_t[0] / (1 - p_t[1] - p_t[2]) * 252), color="black", linewidth=0.8, label="long-run volatility")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Term structure: average annualized volatility over the next H days")
ax.set_xlabel("horizon H (days)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
zs = np.sort(z_hat)
pp = (np.arange(1, zs.size + 1) - 0.5) / zs.size
ax.plot(stats.norm.ppf(pp), zs, ".", markersize=2, label="vs normal")
ax.plot(stats.t.ppf(pp, nu_hat) * np.sqrt((nu_hat - 2) / nu_hat), zs, ".", markersize=2, label=f"vs standardized t({nu_hat:.1f})")
lim = [-6, 6]
ax.plot(lim, lim, color="grey", linestyle=":")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_title("QQ plot of standardized residuals")
ax.set_xlabel("theoretical quantile")
ax.set_ylabel("sample quantile")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 7. Exercises

### Basic

1. For $\omega = 10^{-6}$, $\alpha = 0.05$, $\beta = 0.94$, compute the long-run annual volatility, the persistence and the half-life.
2. With the parameters of exercise 1, today's variance is $2\cdot 10^{-4}$ and today's return is +1%. Compute tomorrow's variance and the forecast for 10 days ahead.
3. A fitted model for a stock index gives $\alpha + \beta = 0.995$. **Interpret** this number for a risk manager who must decide how long to keep a crisis-level
   VaR after a shock.

### Intermediate

4. Fit the Gaussian GARCH with variance targeting (two parameters) and compare the estimates and the log-likelihood with the unrestricted fit.
5. The GJR-GARCH adds a leverage term: $\sigma_t^2 = \omega + (\alpha + \gamma\mathbb{1}_{r_{t-1}<0})r_{t-1}^2 + \beta\sigma_{t-1}^2$. Plot the news impact curve $\sigma_t^2$ as a function of $r_{t-1}$ (with $\sigma_{t-1}^2 = \bar\sigma^2$) for
   GARCH ($\alpha = 0.08$) and GJR ($\alpha = 0.03$, $\gamma = 0.10$), both with $\beta = 0.90$.

### Advanced

6. Likelihood-ratio test: compare GARCH with the restricted model EWMA ($\omega = 0$, $\alpha + \beta = 1$, $\lambda$ estimated). Compute $2(\ell_{\text{GARCH}} - \ell_{\text{EWMA}})$. Why is the usual $\chi^2$
   reference not exactly right here?
7. Fit the Gaussian GARCH to 5 years of data (1,260 days) for 60 simulated histories. How variable is the persistence $\alpha + \beta$, and how often is it estimated
   above 0.995?

## 8. Solutions / Checks

In [ ]:
# 1. and 2.
om, al, be = 1e-6, 0.05, 0.94
v_bar = om / (1 - al - be)
print(f"1. long-run vol {np.sqrt(v_bar * 252):.1%}, persistence {al + be:.2f}, half-life {np.log(0.5) / np.log(al + be):.0f} days")
s2_tom = om + al * 0.01**2 + be * 2e-4
print(f"2. tomorrow {s2_tom:.3e} ({np.sqrt(s2_tom):.2%}), in 10 days {v_bar + (al + be) ** 9 * (s2_tom - v_bar):.3e}")

# 4.
obj_vt = lambda u: -loglik_normal([r.var() * (1 - to_params([0, u[0], u[1]])[1] - to_params([0, u[0], u[1]])[2]),
                                    to_params([0, u[0], u[1]])[1], to_params([0, u[0], u[1]])[2]], r)
res_vt = minimize(obj_vt, from_params([1e-6, 0.05, 0.9])[1:], method="BFGS")
pa = to_params([0, *res_vt.x])
print(f"4. variance targeting: alpha {pa[1]:.4f}, beta {pa[2]:.4f}, loglik {-res_vt.fun:.1f}  "
      f"(unrestricted: alpha {p_norm[1]:.4f}, beta {p_norm[2]:.4f}, loglik {ll_norm:.1f})")

In [ ]:
# 5.
v_b = 1e-4
rr = np.linspace(-0.05, 0.05, 201)
nic_garch = 2e-6 + 0.08 * rr**2 + 0.90 * v_b
om_gjr = v_b * (1 - 0.03 - 0.10 / 2 - 0.90)
nic_gjr = om_gjr + (0.03 + 0.10 * (rr < 0)) * rr**2 + 0.90 * v_b
plt.figure(figsize=(8, 4))
plt.plot(rr, np.sqrt(nic_garch * 252), label="GARCH")
plt.plot(rr, np.sqrt(nic_gjr * 252), label="GJR-GARCH")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.title("News impact curve: tomorrow's volatility vs today's return")
plt.xlabel("today's return")
plt.ylabel("tomorrow's volatility (annualized)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 6.
lam_grid = np.linspace(0.85, 0.995, 59)
ll_e = [loglik_normal_ewma for loglik_normal_ewma in
        [-0.5 * np.sum(np.log(2 * np.pi) + np.log(ewma_variance(r, l)) + r**2 / ewma_variance(r, l)) for l in lam_grid]]
lam_best = lam_grid[int(np.argmax(ll_e))]
LR = 2 * (ll_norm - max(ll_e))
print(f"6. best EWMA lambda {lam_best:.3f}; LR statistic {LR:.1f} (chi2 with 2 df, 99% quantile {stats.chi2.ppf(0.99, 2):.1f})")

In [ ]:
# 7.
rng7 = np.random.default_rng(11)
pers = np.array([sum(fit_garch(simulate_garch(omega0, alpha0, beta0, 1_260, rng7, nu0)[0], "normal")[0][1:]) for _ in range(60)])
print(f"7. persistence over 5-year samples: mean {pers.mean():.3f}, 5%-95% range {np.quantile(pers, 0.05):.3f} to {np.quantile(pers, 0.95):.3f}, "
      f"above 0.995 in {np.mean(pers > 0.995):.0%}")

**1.** $\bar\sigma^2 = 10^{-6}/0.01 = 10^{-4}$, long-run volatility 15.9%; persistence 0.99; half-life $\ln 0.5/\ln 0.99 = 69$ days.

**2.** $\sigma^2_{t+1} = 10^{-6} + 0.05\cdot 10^{-4} + 0.94\cdot 2\cdot 10^{-4} = 1.94\cdot 10^{-4}$ (1.39%); in 10 days $10^{-4} + 0.99^9\cdot 0.94\cdot 10^{-4} = 1.86\cdot 10^{-4}$.

**3.** A persistence of 0.995 means a half-life of about 140 trading days: after a shock, the model expects volatility to stay elevated for
many months, and VaR will come down slowly. The risk manager should also know that persistence estimates close to 1 are imprecise and
partly reflect changes in the long-run level over the sample (exercise 7); a slow decay in the model is not proof that the crisis regime will last.

**4.** Variance targeting gives estimates close to the unrestricted ones and a log-likelihood only slightly lower: one parameter less at almost
no cost, and the long-run volatility is fixed at the sample value.

**5.** The GARCH curve is symmetric; the GJR curve rises much more for negative returns: a −3% day raises tomorrow's volatility more than a +3% day,
which matches equity data (leverage effect).

**6.** The LR statistic is 120, far above 9.2: the data reject EWMA in favour of a mean-reverting GARCH. The $\chi^2_2$ reference is only approximate because the
restriction $\alpha + \beta = 1$ lies on the boundary of the stationary region and $\omega = 0$ on the boundary of the parameter space.

**7.** With five years of data the persistence estimates average 0.973 (true 0.98), with a 5–95% range of 0.936 to 0.992, and exceed 0.995 in 2% of
histories. Twenty years give a standard error of about 0.006. The long-run level and the speed of mean reversion are hard to pin down in
short samples, the same lesson as for means in module 06.

## 9. Conclusion

- GARCH(1,1) adds a constant to the EWMA recursion: $\sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta\sigma_{t-1}^2$, with long-run variance $\omega/(1-\alpha-\beta)$ and persistence $\alpha + \beta$.
- Forecasts revert to the long-run level at rate $(\alpha+\beta)^h$, giving a term structure of volatility; EWMA is the non-reverting limit.
- The parameters are estimated by maximizing the Gaussian or Student-$t$ log-likelihood numerically; the Hessian gives standard errors, and
  $\alpha$, $\beta$ are strongly negatively correlated.
- Standardized residuals should be uncorrelated in their squares; their remaining fat tails call for $t$ innovations.
- A GARCH-$t$ VaR has close to the nominal exceedance rate; normal-quantile VaRs exceed it.

This closes module 09. Next: [Module 10, 01 Binomial tree and risk-neutral pricing](../10_stochastic_processes_and_pricing/01_binomial_tree_and_risk_neutral_pricing.ipynb).